# Kafka Producer — 模拟实时出租车数据流

读取 **ODS 层**原始数据（未清洗）→ 逐条发送到 Kafka topic `tlc-trips`，模拟出租车调度系统实时上报行程。

使用 ODS 而非 DWD 的原因：实时链路需要自己做清洗，对账结果才能反映两条链路的口径差异。
若从 fact_trips（已清洗）回放，实时与离线完全相同，差异恒为 0，对账没有意义。

## 1. 导入依赖

In [1]:
import json
import time
import os
import pandas as pd
from kafka import KafkaProducer
from datetime import datetime

print('依赖导入成功')

依赖导入成功


## 2. 初始化 Kafka Producer

In [2]:
producer = KafkaProducer(
    bootstrap_servers='localhost:9092',
    value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8'),
    key_serializer=lambda k: str(k).encode('utf-8')
)

TOPIC = 'tlc-trips'
print(f'Producer 连接成功，目标 topic: {TOPIC}')

Producer 连接成功，目标 topic: tlc-trips


## 3. 读取 ODS 层原始数据（Yellow，2024-03-15）

In [ ]:
# 读 ODS 层：未清洗的原始数据，字段名已在 ODS ingest 时统一（tpep → pickup_datetime）
ODS_PATH = os.path.expanduser(
    '~/tlc-spark-pipeline/data/warehouse/ods/yellow'
    '/pickup_year=2024/pickup_month=3/pickup_day=15'
)

df = pd.read_parquet(ODS_PATH)

# 按事件时间（pickup_datetime）排序，模拟真实数据到达顺序
df = df.sort_values('pickup_datetime').reset_index(drop=True)

print(f'读取记录数: {len(df):,}')
print(f'时间范围: {df["pickup_datetime"].min()} ~ {df["pickup_datetime"].max()}')
print(f'字段列表: {list(df.columns)}')
df[['pickup_datetime', 'PULocationID', 'DOLocationID', 'fare_amount', 'passenger_count']].head()

## 4. 发送数据到 Kafka

In [ ]:
SEND_LIMIT = 5000
DELAY_SECONDS = 0.02   # 20ms 间隔，5000 条约 100 秒发完

sent = 0
errors = 0

for idx, row in df.head(SEND_LIMIT).iterrows():
    # 发送原始 ODS 字段，不包含 DWD 派生列（time_of_day / trip_duration_min 等）
    # 实时消费端自行清洗 + 派生，才能与离线结果产生可对比的口径差异
    record = {
        'VendorID':         int(row['VendorID'])         if pd.notna(row.get('VendorID')) else None,
        'pickup_datetime':  str(row['pickup_datetime']),
        'dropoff_datetime': str(row['dropoff_datetime']),
        'PULocationID':     int(row['PULocationID']),
        'DOLocationID':     int(row['DOLocationID']),
        'passenger_count':  int(row['passenger_count'])  if pd.notna(row.get('passenger_count')) else None,
        'trip_distance':    float(row['trip_distance']),
        'fare_amount':      float(row['fare_amount']),
        'tip_amount':       float(row['tip_amount']),
        'total_amount':     float(row['total_amount']),
        'payment_type':     int(row['payment_type'])     if pd.notna(row.get('payment_type')) else None,
        'source_type':      str(row['source_type']),
        'event_timestamp':  str(row['pickup_datetime']),   # 供 Spark Streaming 事件时间用
    }

    try:
        producer.send(TOPIC, key=record['PULocationID'], value=record)
        sent += 1
        if sent % 500 == 0:
            print(f'已发送 {sent} 条 | {record["pickup_datetime"]} | 区域 {record["PULocationID"]}→{record["DOLocationID"]} | ${record["fare_amount"]}')
        time.sleep(DELAY_SECONDS)
    except Exception as e:
        errors += 1
        print(f'发送失败: {e}')

producer.flush()
print(f'\n完成！成功 {sent} 条，失败 {errors} 条')

## 5. 验证 — 从 Kafka 读回几条消息

In [5]:
from kafka import KafkaConsumer

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers='localhost:9092',
    auto_offset_reset='earliest',      # 从最早的消息开始读
    enable_auto_commit=False,
    value_deserializer=lambda m: json.loads(m.decode('utf-8')),
    consumer_timeout_ms=3000           # 3秒没有新消息就停止
)

print('读回前 5 条消息验证：\n')
for i, msg in enumerate(consumer):
    if i >= 5:
        break
    v = msg.value
    print(f'Partition {msg.partition} | Offset {msg.offset}')
    print(f'  {v["pickup_datetime"]}  区域 {v["PULocationID"]}→{v["DOLocationID"]}  ${v["fare_amount"]}  {v["time_of_day"]}')
    print()

consumer.close()
producer.close()

读回前 5 条消息验证：



/var/folders/r1/85vrv9s95n3cdht963956czw0000gn/T/ipykernel_6440/289090040.py:3: DeprecationWarning: value_deserializer does not implement kafka.serializer.Deserializer
  consumer = KafkaConsumer(


Partition 1 | Offset 1345
  2024-01-01 00:00:03  区域 136→18  $5.1  late_night

Partition 1 | Offset 1346
  2024-01-01 00:00:11  区域 113→234  $16.3  late_night

Partition 1 | Offset 1347
  2024-01-01 00:01:22  区域 170→162  $5.1  late_night

Partition 1 | Offset 1348
  2024-01-01 00:01:58  区域 239→142  $5.1  late_night

Partition 1 | Offset 1349
  2024-01-01 00:02:16  区域 142→24  $12.1  late_night

